# Comparación con NLP: spaCy y Transformers

**Responsable:** Persona 1

Se procesa el conjunto de prueba (`data/test_set.csv`) con dos herramientas de NLP para español y se
guardan las salidas en el formato acordado (`docs/gramatica_es.md`, sección 2.4):

| Herramienta | Salida |
| --- | --- |
| spaCy `es_core_news_sm` | `results/spacy.jsonl` |
| Modelo de HuggingFace Transformers para español | `results/transformers.jsonl` |

spaCy no decide si la oración es válida (`acepta` = `null`): se compara por **etiquetado** y por
**funciones sintácticas** (sujeto, verbo y objeto). Transformers etiqueta con un modelo y **juzga la
gramaticalidad** con otro (BETO, sección 2.5), así que sí llena `acepta`. La conversión a nuestras categorías es la de
la sección 2.5, implementada en `experiments/tagsets.py`.

**Antes de correrlo** (una sola vez, con el `.venv` del proyecto activado):

```bash
pip install -r requirements.txt -r requirements-experiments.txt
python -m spacy download es_core_news_sm
```

Preguntas que este notebook debe responder (van al informe):

1. ¿En qué oraciones spaCy y Transformers analizan bien lo que la CFG **no puede**? Pista: `en_gramatica = no` y `espanol_valido = si`.
2. ¿En qué oraciones la CFG acierta y ellos **no**?

In [ ]:
# Configuración: el notebook vive en notebooks/, el código del proyecto en la raíz.
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from evaluation.formats import Prediction, load_predictions, load_test_set, results_path, save_predictions

pd.set_option("display.max_colwidth", 80)
items = load_test_set()
by_id = {item.id: item for item in items}
print(f"{len(items)} oraciones en el conjunto de prueba")

In [ ]:
pd.DataFrame([{
    "id": i.id, "nivel": i.level, "oracion": i.sentence,
    "en_gramatica": i.in_grammar, "espanol_valido": i.valid_spanish,
    "tipo_error": i.error_type, "fenomeno": ";".join(i.phenomena),
} for i in items])

## 1. spaCy

### 1.1 Correr spaCy y guardar `results/spacy.jsonl`

In [ ]:
import spacy

try:
    nlp = spacy.load("es_core_news_sm")
except OSError:
    raise SystemExit("Falta el modelo: python -m spacy download es_core_news_sm")

# Anotar en el informe: versión de spaCy y del modelo.
print("spaCy", spacy.__version__, "| modelo", nlp.meta["lang"] + "_" + nlp.meta["name"], nlp.meta["version"])

In [ ]:
from experiments.tagsets import spacy_output

docs = {}  # id -> Doc de spaCy, para el análisis de abajo
spacy_predictions = []
for item in items:
    start = time.perf_counter()
    try:
        doc = nlp(item.sentence)
        docs[item.id] = doc
        row = spacy_output(item.id, doc, round((time.perf_counter() - start) * 1000, 3))
        spacy_predictions.append(Prediction.from_json(row))
    except Exception as e:  # el formato pide escribir la línea aunque falle
        spacy_predictions.append(Prediction(item.id, "spacy", accepts=None,
                                            run_error=f"{type(e).__name__}: {e}"))

print("Guardado en", save_predictions("spacy", spacy_predictions, items))

### 1.2 Etiquetas y funciones sintácticas, oración por oración

Compara la salida de spaCy (ya convertida a nuestras categorías) con lo esperado en el conjunto de prueba.
Las métricas oficiales las calcula la Persona 4 con `evaluation/metrics.py`; esta tabla es para leer los
casos uno por uno.

In [ ]:
rows = []
for item, pred in zip(items, spacy_predictions):
    expected = [c for _, c in item.tags]
    got = [c for _, c in (pred.tags or [])]
    wrong = [f"{w}: {e}→{g}" for (w, e), g in zip(item.tags, got) if e != g] if len(got) == len(expected) else ["distinto número de palabras"]
    rows.append({
        "id": item.id,
        "oracion": item.sentence,
        "en_gramatica": item.in_grammar,
        "etiquetas_ok": not wrong,
        "errores_etiqueta": ", ".join(wrong),
        "sujeto": f"{pred.subject!r} (esperado {item.subject!r})",
        "verbo": f"{pred.verb!r} (esperado {item.verb!r})",
        "objeto": f"{pred.object!r} (esperado {item.object!r})",
        "funciones_ok": (pred.subject, pred.verb, pred.object) == (item.subject, item.verb, item.object),
    })
table = pd.DataFrame(rows)
table

In [ ]:
total_words = sum(len(i.tags) for i in items)
right_words = sum(
    sum(e == g for (_, e), (_, g) in zip(i.tags, p.tags))
    for i, p in zip(items, spacy_predictions) if p.tags and len(p.tags) == len(i.tags)
)
print(f"Etiquetas correctas: {right_words}/{total_words} palabras")
print(f"Oraciones con todas las etiquetas correctas: {table.etiquetas_ok.sum()}/{len(table)}")
print(f"Oraciones con sujeto, verbo y objeto correctos: {table.funciones_ok.sum()}/{len(table)}")
for in_grammar, group in table.groupby("en_gramatica"):
    print(f"  en_gramatica={in_grammar}: etiquetas OK {group.etiquetas_ok.sum()}/{len(group)}, "
          f"funciones OK {group.funciones_ok.sum()}/{len(group)}")

### 1.3 Análisis completo de spaCy

Lo que spaCy ve antes de convertirlo a nuestras categorías: etiqueta UPOS, relación de dependencia, núcleo y
rasgos morfológicos. Sirve para explicar los errores de la tabla anterior.

In [ ]:
def spacy_detail(doc) -> pd.DataFrame:
    return pd.DataFrame([{
        "palabra": t.text, "upos": t.pos_, "dep": t.dep_, "nucleo": t.head.text,
        "genero": ",".join(t.morph.get("Gender")), "numero": ",".join(t.morph.get("Number")),
        "rasgos": str(t.morph),
    } for t in doc if t.pos_ != "PUNCT"])


for item in items:
    print(f"\n{item.id}. {item.sentence}   (en_gramatica={item.in_grammar})")
    print(spacy_detail(docs[item.id]).to_string(index=False))

### 1.4 Experimento: ¿podría spaCy detectar los errores de concordancia?

spaCy no rechaza oraciones, pero sí da rasgos de género y número. Aquí se prueba si con esos rasgos se
podrían detectar los mismos errores que detecta nuestro analizador semántico, con las reglas de la
sección 1.3:

- `det` y `amod` deben concordar en género y número con su núcleo;
- el sujeto (`nsubj`) debe concordar en número con el verbo raíz.

Es un experimento de análisis: **no cambia `results/spacy.jsonl`**, donde `acepta` sigue siendo `null`
como dice el formato acordado.

In [ ]:
def spacy_agreement_problems(doc) -> list[str]:
    """Desacuerdos que se pueden ver con los rasgos de spaCy. Si falta un rasgo, no se puede comparar."""
    problems = []
    for t in doc:
        if t.dep_ in ("det", "amod"):
            for feature in ("Gender", "Number"):
                a, b = t.morph.get(feature), t.head.morph.get(feature)
                if a and b and a != b:
                    problems.append(f"{t.text}/{t.head.text}: {feature} {a[0]}≠{b[0]}")
        if t.dep_ == "nsubj" and t.head.dep_ == "ROOT" and t.head.pos_ in ("VERB", "AUX"):
            a, b = t.morph.get("Number"), t.head.morph.get("Number")
            if a and b and a != b:
                problems.append(f"{t.text}/{t.head.text}: sujeto-verbo {a[0]}≠{b[0]}")
    return problems


rows = []
for item in items:
    problems = spacy_agreement_problems(docs[item.id])
    rows.append({
        "id": item.id, "oracion": item.sentence,
        "error_esperado": item.error_type or "ninguno",
        "spacy_detecta": "; ".join(problems) or "nada",
        "coincide": (item.error_type == "concordancia") == bool(problems),
    })
pd.DataFrame(rows)

### 1.5 Ambigüedad de las frases preposicionales

Nuestra gramática pega cada PP al sustantivo más cercano (sección 1.4 del documento). spaCy decide con su
modelo: si la preposición depende de un sustantivo (`nmod`), la pega al sintagma nominal; si depende del
verbo (`obl`), la pega al sintagma verbal.

In [ ]:
rows = []
for item in items:
    doc = docs[item.id]
    for t in doc:
        if t.dep_ == "case" and t.pos_ == "ADP":
            phrase = t.head                       # núcleo del sintagma preposicional
            attached_to = phrase.head
            rows.append({
                "id": item.id, "oracion": item.sentence,
                "pp": " ".join(x.text for x in phrase.subtree),
                "spacy_la_une_a": f"{attached_to.text} ({phrase.dep_})",
                "spacy": "verbo" if attached_to.pos_ in ("VERB", "AUX") else "sustantivo",
                "nuestra_gramatica": "sustantivo más cercano",
            })
pd.DataFrame(rows) if rows else "No hay frases preposicionales en el conjunto de prueba."

### 1.6 Sondeo de robustez

Variaciones mínimas de algunas oraciones, para separar lo que spaCy "sabe" de los accidentes del modelo:
la mayúscula inicial, oraciones muy cortas y pares mínimos correcto/incorrecto. No son parte del conjunto
de prueba.

In [ ]:
PROBES = [
    ("mayúscula inicial", "Come la manzana."),
    ("mayúscula inicial", "come la manzana."),
    ("sujeto explícito", "Ella come la manzana."),
    ("oración corta, incorrecta", "Los perros come."),
    ("oración corta, correcta", "Los perros comen."),
    ("oración corta, correcta", "El perro come."),
    ("concordancia, incorrecta", "La elefante grises duerme."),
    ("concordancia, correcta", "El elefante gris duerme."),
]

pd.DataFrame([{
    "caso": case, "oracion": sentence,
    "analisis": " ".join(f"{t.text}/{t.pos_}/{t.dep_}" for t in nlp(sentence) if t.pos_ != "PUNCT"),
} for case, sentence in PROBES])

### 1.7 Matriz de confusión y reporte de clasificación

Métricas de **etiquetado** palabra por palabra (sección 2.7 b) y de **funciones sintácticas** (2.7 c).
spaCy no decide si la oración es válida (`acepta` = `null`), así que no tiene matriz de aceptación: esa
está en las secciones 2.5 y 3 para los sistemas que sí deciden (parser, Transformers y LLM).

Las oraciones con un número distinto de palabras no se pueden comparar palabra por palabra; se
excluyen y se reportan aparte. Las figuras y tablas se guardan en `results/figures/` y `results/reports/`
para el informe.

In [ ]:
import matplotlib.pyplot as plt

from experiments import reports

FIGURES = ROOT / "results" / "figures"
REPORTS = ROOT / "results" / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

y_true, y_pred, skipped = reports.tag_pairs(items, spacy_predictions)
print(f"Palabras comparadas: {len(y_true)} | oraciones excluidas por distinto número de palabras: {skipped or 'ninguna'}")

spacy_confusion = reports.tag_confusion(y_true, y_pred)
spacy_confusion

In [ ]:
reports.plot_confusion(spacy_confusion, "spaCy: categorías esperadas vs. predichas",
                       FIGURES / "spacy_matriz_confusion.png")
plt.show()

In [ ]:
spacy_report = reports.tag_report(y_true, y_pred)
spacy_report.to_csv(REPORTS / "spacy_reporte_clasificacion.csv")
spacy_report.round(3)

In [ ]:
spacy_functions = reports.function_report(items, spacy_predictions)
spacy_functions.to_csv(REPORTS / "spacy_funciones.csv")
spacy_functions

In [ ]:
# Resumen con todas las métricas de spaCy, también por nivel.
def spacy_summary(subset, name):
    ids = {i.id for i in subset}
    preds = [p for p in spacy_predictions if p.id in ids]
    yt, yp, skip = reports.tag_pairs(subset, preds)
    rep = reports.tag_report(yt, yp) if yt else None
    fun = reports.function_report(subset, preds)
    times = [p.time_ms for p in preds if not p.run_error]
    return {
        "nivel": name,
        "oraciones": len(subset),
        "palabras": len(yt),
        "exactitud_etiquetas": rep.loc["exactitud", "f1"] if rep is not None else None,
        "precision_macro": rep.loc["promedio macro", "precision"] if rep is not None else None,
        "cobertura_macro": rep.loc["promedio macro", "cobertura"] if rep is not None else None,
        "f1_macro": rep.loc["promedio macro", "f1"] if rep is not None else None,
        "f1_ponderado": rep.loc["promedio ponderado", "f1"] if rep is not None else None,
        "excluidas": len(skip),
        "sujeto": fun.loc["sujeto", "exactitud"],
        "verbo": fun.loc["verbo", "exactitud"],
        "objeto": fun.loc["objeto", "exactitud"],
        "tiempo_ms_promedio": sum(times) / len(times) if times else None,
    }


spacy_all = pd.DataFrame(
    [spacy_summary(items, "todos")]
    + [spacy_summary([i for i in items if i.level == lv], str(lv)) for lv in (1, 2, 3)]
).set_index("nivel")
spacy_all.to_csv(REPORTS / "spacy_resumen.csv")
spacy_all.round(3)

### 1.8 Hallazgos preliminares de spaCy

> Escritos con las 8 oraciones de ejemplo del conjunto de prueba y spaCy 3.8 con `es_core_news_sm` 3.8.0.
> **Volver a revisarlos cuando la Persona 4 complete el conjunto de prueba.**

1. **En las oraciones que nuestra gramática genera, spaCy acierta.** Las 5 tienen etiquetas, sujeto,
   verbo y objeto correctos.
2. **No detecta errores de concordancia, ni siquiera con sus rasgos.** En *"La elefante grises duerme."*
   etiqueta *elefante* como nombre propio (`PROPN`) sin género ni número, así que no hay nada que
   comparar con *la* ni con *grises*. Nuestra CFG con rasgos sí la rechaza.
3. **Sus fallos en oraciones cortas no tienen que ver con la concordancia.** En *"Los perros come."*
   etiqueta *come* como sustantivo, pero en *"Los perros comen."* (correcta) también falla: etiqueta
   *comen* como sustantivo y *perros* como raíz. El modelo pequeño es frágil con oraciones de sujeto y
   verbo sin objeto.
4. **Por la mayúscula inicial, no analiza bien el sujeto omitido.** *"Come la manzana."* sale con *Come*
   como nombre propio, pero *"come la manzana."* sale bien: verbo como raíz y *manzana* como objeto. Es
   justo un caso que la CFG **no puede** generar (sujeto omitido) y spaCy **sí** podría, si no fuera por
   la mayúscula.
5. **Ambigüedad de las PP: spaCy elige una lectura distinta a la nuestra.** En *"Él mató un elefante en
   sus pijamas"* y *"Mi niña comió la manzana con su perro"* une la PP al **verbo** (`obl`). Nuestro
   parser la une al **sustantivo más cercano**. Ninguno muestra que existe otra lectura: la CFG porque
   el parser elige una, y spaCy porque siempre devuelve un solo árbol.
6. **Matriz de confusión (sección 1.7): el único error de etiquetado es Verb → Noun.** De 40 palabras
   acierta 38 (exactitud 0,95). Las dos que falla son *come* en *"Los perros come."* y en *"Come la
   manzana."*, que son los hallazgos 3 y 4. Por eso `Verb` tiene cobertura 0,75 y `Noun` precisión 0,85;
   todas las demás categorías tienen 1,0. Por nivel, el 3 baja a 0,5 en sujeto, verbo y objeto por
   *"Come la manzana."* (spaCy toma *manzana* como verbo raíz). Con tan pocas oraciones, cada error
   mueve mucho los porcentajes.
7. **Tiempo:** unos 2 ms por oración. La primera oración tarda más (unos 9 ms) porque incluye la
   preparación del modelo, por eso el promedio del nivel 1 sale más alto.
8. **Errores que no cambian las métricas:** *come* con modo subjuntivo (`Mood=Sub`) en *"Ella come una
   manzana."*, *ayer* unido a *gris* en vez de al verbo, y *pijamas* marcado como femenino.

TODO(P1): probar un modelo más grande (`es_core_news_md`, `es_core_news_lg` o el de transformers,
`es_dep_news_trf`) para ver si los hallazgos 3 y 4 son límites del modelo pequeño o de spaCy en general.

## 2. HuggingFace Transformers

Se prueban **dos** modelos de clasificación de tokens para español que dan etiquetas **UPOS** (así se
usa la misma conversión de la sección 2.5):

| Clave | Modelo | Base | Parámetros | Pesos | Documentación |
| --- | --- | --- | --- | --- | --- |
| `xlmr_ud28` | `wietsedv/xlm-roberta-base-ft-udpos28-es` | XLM-RoBERTa base (multilingüe) | 277 M | 1,1 GB | Universal Dependencies 2.8; exactitud 95,5 % en el test de español (ficha del modelo); Apache-2.0 |
| `beto16` | `mrm8488/bert-spanish-cased-finetuned-pos-16-tags` | BETO (BERT en español) | 109 M | 440 MB | Sin ficha: no se publican los datos de entrenamiento ni la exactitud |

**Oficial:** el formato acordado tiene un solo `results/transformers.jsonl`. Se usa `xlmr_ud28` porque es el
que se puede documentar en el informe; `beto16` se guarda en `results/reports/` y se compara aquí. Para
cambiarlo, editar `OFFICIAL` abajo.

**Cómo se etiqueta:** se le pasan al modelo las palabras ya separadas y cada palabra toma la etiqueta de
su primer fragmento (`experiments/hf_tagger.py`). Así hay exactamente una etiqueta por palabra; el
`pipeline` de HuggingFace con agrupación puede juntar palabras vecinas con la misma etiqueta.

**Recursos (medidos en una Mac con 8 GB):** se carga un modelo a la vez; el proceso llega a ~1 GB de RAM.
La primera vez cada modelo se descarga a `~/.cache/huggingface`. Instalar antes `torch` y `transformers`
(`requirements-experiments.txt`).

In [ ]:
import importlib.util

MODELS = {
    # clave: (nombre en el Hub, use_safetensors)
    "xlmr_ud28": ("wietsedv/xlm-roberta-base-ft-udpos28-es", False),   # solo publica pytorch_model.bin
    "beto16": ("mrm8488/bert-spanish-cased-finetuned-pos-16-tags", None),
}
OFFICIAL = "xlmr_ud28"

TRANSFORMERS_READY = all(importlib.util.find_spec(m) for m in ("torch", "transformers"))
if not TRANSFORMERS_READY:
    print("Falta instalar torch y transformers (requirements-experiments.txt); se salta esta sección.")

In [ ]:
from evaluation.formats import write_jsonl
from experiments.hf_tagger import HFTagger
from experiments.tagsets import upos_category

# Variantes para el sondeo de robustez (mayúscula inicial, oraciones cortas).
HF_PROBES = [
    "Come la manzana", "come la manzana", "Él mató un elefante", "él mató un elefante",
    "Los perros come", "Los perros comen", "El perro come", "La elefante grises duerme",
]

hf_predictions, hf_info, hf_probe_rows = {}, {}, []
if TRANSFORMERS_READY:
    for key, (model_name, use_safetensors) in MODELS.items():
        tagger = HFTagger(model_name, use_safetensors=use_safetensors)   # un modelo a la vez
        hf_info[key] = {"modelo": model_name, "parametros_M": round(tagger.parameters / 1e6),
                        "carga_s": round(tagger.load_seconds, 1)}
        predictions = []
        for item in items:
            words = item.sentence.rstrip().rstrip(".").split()
            start = time.perf_counter()
            try:
                raw = tagger.tag(words)
                tags = [[w.lower(), upos_category(t, w) or "X"] for w, t in zip(words, raw)]
                predictions.append(Prediction(item.id, "transformers", accepts=None, tags=tags,
                                              time_ms=round((time.perf_counter() - start) * 1000, 3),
                                              ))
            except Exception as e:  # el formato pide escribir la línea aunque falle
                predictions.append(Prediction(item.id, "transformers", accepts=None,
                                              run_error=f"{type(e).__name__}: {e}"))
        hf_predictions[key] = predictions
        for probe in HF_PROBES:
            words = probe.split()
            hf_probe_rows.append({"modelo": key, "oracion": probe,
                                  "etiquetas": " ".join(f"{w}/{t}" for w, t in zip(words, tagger.tag(words)))})
        tagger.close()
        del tagger

    print("Oficial:", save_predictions("transformers", hf_predictions[OFFICIAL], items))
    for key, predictions in hf_predictions.items():
        if key != OFFICIAL:
            path = REPORTS / f"transformers_{key}.jsonl"
            write_jsonl(path, (p.to_json() for p in predictions))
            print("Comparación:", path)
    display(pd.DataFrame(hf_info).T)

### 2.1 Matriz de confusión y reporte de clasificación de cada modelo

In [ ]:
hf_reports = {}
for key, predictions in hf_predictions.items():
    yt, yp, skip = reports.tag_pairs(items, predictions)
    print(f"\n=== {key} ({MODELS[key][0]}) | palabras {len(yt)} | excluidas {skip or 'ninguna'}")
    matrix = reports.tag_confusion(yt, yp)
    display(matrix)
    reports.plot_confusion(matrix, f"{key}: categorías esperadas vs. predichas",
                           FIGURES / f"transformers_{key}_matriz_confusion.png")
    plt.show()
    hf_reports[key] = reports.tag_report(yt, yp)
    hf_reports[key].to_csv(REPORTS / f"transformers_{key}_reporte_clasificacion.csv")
    display(hf_reports[key].round(3))

### 2.2 Comparación de los tres etiquetadores

Etiquetado palabra por palabra contra el conjunto de prueba. Los dos modelos de Transformers solo
etiquetan (no dan sujeto, verbo ni objeto).

In [ ]:
def tagger_summary(name, predictions):
    yt, yp, skip = reports.tag_pairs(items, predictions)
    rep = reports.tag_report(yt, yp)
    times = [p.time_ms for p in predictions if not p.run_error]
    wrong = [f"{w}: {e}→{g}" for i in items for p in predictions if p.id == i.id and p.tags
             and len(p.tags) == len(i.tags) for (w, e), (_, g) in zip(i.tags, p.tags) if e != g]
    return {"sistema": name, "exactitud": rep.loc["exactitud", "f1"],
            "precision_macro": rep.loc["promedio macro", "precision"],
            "cobertura_macro": rep.loc["promedio macro", "cobertura"],
            "f1_macro": rep.loc["promedio macro", "f1"], "excluidas": len(skip),
            # Mediana: la primera oración incluye la preparación del modelo (~2-3 s) y distorsiona el promedio.
            "ms_por_oracion_mediana": float(pd.Series(times).median()) if times else None,
            "errores": "; ".join(wrong) or "ninguno"}


comparison = pd.DataFrame(
    [tagger_summary("spacy", spacy_predictions)]
    + [tagger_summary(key, preds) for key, preds in hf_predictions.items()]
).set_index("sistema")
comparison.to_csv(REPORTS / "etiquetadores_comparacion.csv")
comparison.round(3)

In [ ]:
# Oración por oración: ¿quién acierta todas las etiquetas?
def all_tags_ok(item, pred):
    return pred.tags is not None and [c for _, c in pred.tags] == [c for _, c in item.tags]

taggers = {"spacy": spacy_predictions, **hf_predictions}
pd.DataFrame([
    {"id": i.id, "oracion": i.sentence, "en_gramatica": i.in_grammar,
     **{name: "✓" if all_tags_ok(i, next(p for p in preds if p.id == i.id)) else "✗"
        for name, preds in taggers.items()}}
    for i in items
])

### 2.3 Sondeo de robustez (Transformers)

Las mismas variantes que con spaCy (sección 1.6), con las etiquetas UPOS crudas de cada modelo.

In [ ]:
pd.DataFrame(hf_probe_rows).pivot(index="oracion", columns="modelo", values="etiquetas").loc[HF_PROBES]

### 2.4 Hallazgos preliminares de Transformers

> Escritos con las 8 oraciones de ejemplo, torch 2.15 y transformers 5.24. **Volver a revisarlos
> cuando la Persona 4 complete el conjunto de prueba.**

1. **Exactitud parecida, errores distintos.** XLM-R acierta 39 de 40 palabras (0,975); spaCy y BETO,
   38 de 40 (0,95). Pero cada uno falla en palabras diferentes:
   - spaCy: *come* como sustantivo en *"Los perros come."* y en *"Come la manzana."*.
   - XLM-R (`xlmr_ud28`): *Come* como preposición (`ADP`) en *"Come la manzana."*.
   - BETO (`beto16`): *Él* como nombre propio (`PROPN`) en las oraciones 3 y 4.
   En F1 macro el orden cambia (spaCy 0,975, XLM-R 0,970, BETO 0,935): un solo error de XLM-R crea un
   falso `Prep`, y con tan pocos casos por categoría eso pesa mucho. Con 8 oraciones, estas diferencias
   no alcanzan para decir que un sistema es mejor que otro.
2. **Los dos modelos de Transformers aciertan donde falla spaCy en oraciones cortas.** *"Los perros
   come."*, *"Los perros comen."* y *"El perro come."* salen con `VERB` en ambos; el modelo pequeño
   de spaCy fallaba incluso en la correcta.
3. **La mayúscula inicial no afecta igual a todos.**
   - BETO etiqueta *Él* como `PROPN`, pero *él* (minúscula) como `PRON`: es la mayúscula.
   - XLM-R etiqueta *come* como `ADP` **también en minúscula** (*"come la manzana"*): su problema
     es el verbo al inicio sin sujeto, no la mayúscula.
   - BETO etiqueta bien *"Come la manzana"*: es el único de los tres que acierta en la oración de
     sujeto omitido, justo el caso que la CFG no puede generar.
4. **Ninguno detecta concordancia:** solo etiquetan. *"La elefante grises duerme."* y *"Los perros
   come."* salen con etiquetas correctas, como si fueran oraciones válidas. Rechazarlas es justamente
   lo que aporta la CFG con rasgos.
5. **Costo:** la primera oración tarda ~2–3 s (preparación del modelo); después, la mediana es de unas
   decenas de milisegundos por oración en CPU, un orden de magnitud más que spaCy (ver
   `ms_por_oracion_mediana` en la tabla 2.2). La carga del modelo tarda ~1–3 minutos la primera vez por la
   descarga, y unos segundos después. Espacio: 1,1 GB (XLM-R) y 440 MB (BETO).

### 2.5 Gramaticalidad con BETO: pseudo-log-verosimilitud

Un etiquetador no rechaza oraciones. Para que Transformers tenga un valor real de `acepta` se usa un
**modelo de lenguaje con máscara** en español, BETO (`dccuchile/bert-base-spanish-wwm-cased`, 110 M de
parámetros, 440 MB), sin entrenar nada (`experiments/acceptability.py`):

1. **Puntuación:** pseudo-log-verosimilitud (PLL; Salazar et al., 2020). Se enmascara cada fragmento de
   la oración y se mide qué probabilidad le da el modelo al fragmento original. Se usa la variante
   *word-l2r* de Kauf e Ivanova (2023), que enmascara también los fragmentos siguientes de la misma
   palabra, y se promedia por fragmento para no castigar las oraciones largas. Mayor = más aceptable.
2. **Umbral:** se calibra con `data/calibration_acceptability.csv`, 12 pares mínimos correcto/incorrecto
   que **no** están en el conjunto de prueba. Se elige el umbral con mayor exactitud en esas oraciones.
3. **Decisión:** `acepta = puntuación >= umbral`. `tipo_error` queda en `null`: el modelo dice qué tan
   natural es la oración, no *por qué* está mal.

Lo que mide BETO es "¿suena a español correcto?", así que su referencia natural es `espanol_valido`.
Contra `en_gramatica` se ve si coincide con nuestras reglas.

In [ ]:
import csv

from experiments.acceptability import PLLScorer, calibrate_threshold

calibration = list(csv.DictReader((ROOT / "data" / "calibration_acceptability.csv").open(encoding="utf-8")))
pll_rows, cal_scores, test_scores = [], [], {}
if TRANSFORMERS_READY:
    scorer = PLLScorer()
    cal_scores = [scorer.score(r["oracion"]) for r in calibration]
    THRESHOLD, cal_accuracy = calibrate_threshold(cal_scores, [r["espanol_valido"] == "si" for r in calibration])
    for item in items:
        start = time.perf_counter()
        test_scores[item.id] = (scorer.score(item.sentence), (time.perf_counter() - start) * 1000)
    scorer.close()
    del scorer
    print(f"Umbral calibrado: {THRESHOLD:.3f} | exactitud en calibración: {cal_accuracy:.3f}")

In [ ]:
# Pares mínimos de calibración: ¿la versión correcta puntúa más alto que la incorrecta?
pairs = {}
for row, score in zip(calibration, cal_scores):
    pairs.setdefault(row["par"], {})[row["espanol_valido"]] = (row["oracion"], score, row["tipo"])
pair_table = pd.DataFrame([
    {"par": k, "tipo": v["si"][2], "correcta": v["si"][0], "punt_correcta": v["si"][1],
     "incorrecta": v["no"][0], "punt_incorrecta": v["no"][1], "gana_la_correcta": v["si"][1] > v["no"][1]}
    for k, v in pairs.items()
])
print(f"La correcta puntúa más alto en {pair_table.gana_la_correcta.sum()}/{len(pair_table)} pares")
pair_table.round(3)

In [ ]:
# Puntuaciones del conjunto de prueba frente al umbral.
pll_table = pd.DataFrame([
    {"id": i.id, "oracion": i.sentence, "puntuacion": test_scores[i.id][0],
     "acepta": test_scores[i.id][0] >= THRESHOLD,
     "espanol_valido": i.valid_spanish, "en_gramatica": i.in_grammar}
    for i in items
]) if test_scores else pd.DataFrame()
if not pll_table.empty:
    pll_table.to_csv(REPORTS / "transformers_pll.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 3.5))
    cal_good = [s for r, s in zip(calibration, cal_scores) if r["espanol_valido"] == "si"]
    cal_bad = [s for r, s in zip(calibration, cal_scores) if r["espanol_valido"] == "no"]
    ax.scatter(cal_good, [2] * len(cal_good), marker="o", label="calibración: correcta")
    ax.scatter(cal_bad, [2] * len(cal_bad), marker="x", label="calibración: incorrecta")
    good = pll_table[pll_table.espanol_valido]
    bad = pll_table[~pll_table.espanol_valido]
    ax.scatter(good.puntuacion, [1] * len(good), marker="o", label="prueba: español válido")
    ax.scatter(bad.puntuacion, [1] * len(bad), marker="x", label="prueba: inválido")
    for _, r in pll_table.iterrows():
        ax.annotate(str(r["id"]), (r["puntuacion"], 1), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
    ax.axvline(THRESHOLD, linestyle="--", color="gray", label=f"umbral {THRESHOLD:.2f}")
    ax.set_yticks([1, 2], ["prueba", "calibración"])
    ax.set_xlabel("PLL promedio por fragmento (mayor = más aceptable)")
    ax.set_ylim(0.5, 2.5)
    ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    fig.savefig(FIGURES / "transformers_pll_umbral.png", dpi=150)
    plt.show()
pll_table.round(3)

In [ ]:
# acepta y tiempo de BETO en el resultado oficial de Transformers (las etiquetas siguen siendo de XLM-R).
if test_scores:
    for pred in hf_predictions[OFFICIAL]:
        score, ms = test_scores[pred.id]
        pred.accepts = score >= THRESHOLD
        pred.error_type = None        # un modelo de lenguaje no sabe por qué una oración está mal
        pred.time_ms = round(pred.time_ms + ms, 3)
    print("Oficial con acepta:", save_predictions("transformers", hf_predictions[OFFICIAL], items))

    for reference in ("valid_spanish", "in_grammar"):
        matrix = reports.acceptance_confusion(items, hf_predictions[OFFICIAL], reference)
        print(f"\nTransformers (BETO) contra {reports.REFERENCES[reference]}")
        display(matrix)
        display(reports.acceptance_report(matrix).to_frame("valor").T)

### 2.6 Hallazgos preliminares de la gramaticalidad con BETO

> Con las 8 oraciones de ejemplo y 24 de calibración. **Volver a revisarlos cuando la Persona 4 complete
> el conjunto de prueba.**

1. **En pares mínimos, BETO distingue siempre.** En los 12 pares de calibración la versión correcta
   puntúa más alto que la incorrecta, tanto en concordancia (*"El perro duerme"* / *"El perro duermen"*)
   como en orden de palabras.
2. **Con un umbral absoluto, falla bastante más.** El mismo umbral que separa bien la calibración
   (exactitud 0,83) rechaza tres oraciones **correctas** del conjunto de prueba: 2, 3 y 4. Contra
   `espanol_valido` queda con precisión 1,0, cobertura 0,5 y rechazo correcto 1,0: cuando acepta,
   acierta, pero rechaza la mitad del español correcto.
3. **Por qué:** la PLL mezcla gramaticalidad con **frecuencia**. Las oraciones rechazadas tienen palabras
   poco frecuentes (*durmieron*, *elefante*, *gris*, *pijamas*), que bajan la puntuación aunque la oración
   sea correcta. Comparar dos versiones de la misma oración cancela ese efecto; un umbral absoluto no.
   Una mejora conocida es normalizar por la frecuencia de cada palabra (SLOR; Lau et al., 2017).
4. **Sigue al español, no a nuestras reglas.** Acepta *"Come la manzana."* (sujeto omitido), que es
   español correcto pero no está en la gramática. Contra `en_gramatica` eso cuenta como falso positivo.
   Es la misma pregunta que el experimento del LLM: ¿reglas o conocimiento del español?
5. **Costo:** BETO pesa 440 MB y tarda ~80 ms por oración en CPU (hace una pasada por cada fragmento de
   la oración).

## 3. Comparación con la CFG

Usa los resultados que existan en `results/`. El del parser lo genera la Persona 2 con
`python -m evaluation.run_parser`.

In [ ]:
systems = [s for s in ("parser", "spacy", "transformers") if results_path(s).exists()]
by_system = {s: {p.id: p for p in load_predictions(s)} for s in systems}
print("Sistemas con resultados:", ", ".join(systems))


def summary(item, pred):
    if pred is None or pred.run_error:
        return "—"
    decision = "" if pred.accepts is None else ("acepta, " if pred.accepts else f"rechaza ({pred.error_type}), ")
    tags = "etiquetas ✓" if pred.tags and [c for _, c in pred.tags] == [c for _, c in item.tags] else "etiquetas ✗"
    return decision + tags


pd.DataFrame([
    {"id": i.id, "oracion": i.sentence, "en_gramatica": i.in_grammar, "espanol_valido": i.valid_spanish,
     **{s: summary(i, by_system[s].get(i.id)) for s in systems}}
    for i in items
])

### Matriz de confusión de aceptación (parser, Transformers y LLM)

Solo para los sistemas que deciden (`acepta` = `true`/`false`). Se compara contra `en_gramatica` (¿sigue
las reglas?) y contra `espanol_valido` (¿sigue su conocimiento del español?), como pide la sección 2.7 a.

In [ ]:
from experiments import reports

deciding = [s for s in ("parser", "transformers", "llm_zero", "llm_few")
            if results_path(s).exists() and any(p.accepts is not None for p in load_predictions(s))]
if not deciding:
    print("Todavía no hay resultados de sistemas que decidan (parser, Transformers o LLM).")
for system in deciding:
    preds = load_predictions(system)
    for reference in ("in_grammar", "valid_spanish"):
        matrix = reports.acceptance_confusion(items, preds, reference)
        print(f"\n{system} contra {reports.REFERENCES[reference]}")
        display(matrix)
        display(reports.acceptance_report(matrix).to_frame("valor").T)

### Conclusiones (para el informe)

TODO(P1), a partir de los hallazgos de la sección 1.8 y de los de Transformers:

- Oraciones que spaCy/Transformers analizan bien y la CFG no puede (sujeto omitido, orden libre,
  subordinadas, palabras fuera del léxico…).
- Oraciones que la CFG acierta y ellos no (concordancia: un etiquetador no las rechaza).
- Diferencia de fondo: reglas explícitas frente a modelos estadísticos.